# Convolutional Neural Network (CNN)

multi-label classification of toxic comments

In [41]:
import pandas as pd

In [42]:
# Load data
train = pd.read_csv("trainset.csv")
val = pd.read_csv("valset.csv")
test = pd.read_csv("testset.csv")

In [43]:
train['comment_text'] = train['comment_text'].astype(str)
val['comment_text'] = val['comment_text'].astype(str)
test['comment_text'] = test['comment_text'].astype(str)

# Tokenization (Natural Language Processing with PyTorch)

In [44]:
# choose MAX_LEN 
train['comment_text'].apply(lambda x: len(x.split())).describe()

count    102124.000000
mean         65.529631
std          97.155477
min           1.000000
25%          16.000000
50%          34.000000
75%          73.000000
max        1403.000000
Name: comment_text, dtype: float64

In [45]:
train[train['comment_text'].apply(lambda x: len(x.split())) == 1250]['comment_text']

10583    damn you u cunt damn you u cunt damn you u cun...
67416    lol lol lol lol lol lol lol lol lol lol lol lo...
84641    suck my cock d suck my cock d suck my cock d s...
86997    die fag die fag die fag die fag die fag die fa...
Name: comment_text, dtype: object

Based on the result of the word counts, using around 100 to 150 MAX_LEN would enough to filter most of the comments.

The outliers that has 1250 word counts does not have 1250 unique words. 

They are repeatitive comments.

In [46]:
# Configuration Variables
MAX_LEN = 150       # Max unique words per comment
MAX_VOCAB = 200000   # Max words in dictionary
BATCH_SIZE = 32     # for pytorch
EMBED_DIM = 100     # Embedding dimension 
# nn.Embeding layer maps the word index to a dense, continuous vector (embedding)
# This vector's values are learned during training to capture the word's meaning and context.
LEARNING_RATE = 0.001
EPOCHS = 5

# sigmoid activation function is used for binary classification
# threshold is used to classify the output as 0 or 1
SIGMOID_THRESHOLD = 0.5

# Label Columns
LABEL_COLUMNS = val.columns[1:]


In [47]:
from collections import Counter
import re

class Vocabulary:
    """
    Create a vocabulary from a list of comments
    self.word2idx: word to index mapping
    self.idx2word: index to word mapping
    """
    def __init__(self, comments, max_size):
        # <PAD> is for padding, <UNK> is for unknown words
        # <PAD> is needed for batch processing in PyTorch 
        # Depends on the model's max index, 
        # the padding will be applied to make it uniform shape
        self.word2idx = {"<PAD>": 0, "<UNK>": 1} 
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        
        # split comments into words list
        all_words = []
        for comment in comments:
            all_words.extend(comment.split())
            
        # Count word frequencies
        # most_common returns a list of (word, count) tuples
        # restric to max_size most common words
        most_common = Counter(all_words).most_common(max_size)
        
        # (integer index, word)
        # start = 2 because 0 and 1 are reserved for <PAD> and <UNK>
        for idx, (word, _) in enumerate(most_common, start=2): 
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            
    def encode(self, comment):
        """
        Tokenize and encode a comment into indices
        Returns:
            list of indices
        """
        
        # split comment into words list
        words = comment.split()
        # convert words to indices assign 1 <UNK> to unknown words
        indices = [self.word2idx.get(w, 1) for w in words] 
        
        # Padding / Truncating to MAX_LEN
        if len(indices) < MAX_LEN:
            indices += [0] * (MAX_LEN - len(indices)) # Pad with 0
        else:
            indices = indices[:MAX_LEN] # Truncate
            
        return indices

vocab = Vocabulary(train['comment_text'].values, MAX_VOCAB)
print("word2idx size:", len(vocab.word2idx))
print("idx2word size:", len(vocab.idx2word))

word2idx size: 167308
idx2word size: 167308


# Wrap the Dataset into PyTorch Dataset

In [48]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class ToxicDataset(Dataset):
    """
    Toxic dataset class
    Inherits from PyTorch's Dataset
    """
    def __init__(self, df, vocab, is_test=False):
        self.df = df
        self.vocab = vocab
        self.is_test = is_test
        self.label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        """
        Get item from dataset using idx (index)
        """
        # Get text and encode it
        text = str(self.df.iloc[idx]['comment_text'])
        encoded_text = self.vocab.encode(text)
        
        # Convert to pytorch tensor
        # nn.Embedding layer requires long
        x = torch.tensor(encoded_text, dtype=torch.long) 
        
        if self.is_test:
            # return without labels
            return x 
        else:
            # convert labels to float so that CNN can process it.
            labels = self.df.iloc[idx][self.label_cols].values.astype(float)
            y = torch.tensor(labels, dtype=torch.float32)
            return x, y

# Create DataLoaders
train_ds = ToxicDataset(train, vocab)
val_ds = ToxicDataset(val, vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

# CNN Model

In [49]:
import torch.nn as nn

class TextCNN(nn.Module):
    """
    TextCNN class implements a 1D-Convolutional Neural Network (1D-CNN) for text classification.
    """
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(TextCNN, self).__init__()
        
        # Embedding Layer
        # create word embeddings (vectors)
        # vocab_size: total number of unique tokens
        # embed_dim: The size of the vector for each word
        # padding_idx: index of padding token
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        # Convolution Layers (Parallel)
        # Core of the CNN
        # in_channels: number of input channels (embed_dim)
        # out_channels: number of output channels (filters)
        # kernel_size: size of the n-gram (window size)
        # n-gram : contiguous sequence of n words to capture in local context
        self.conv1 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=1)
        self.conv2 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=2)
        self.conv3 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=3)
        self.conv4 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=4)
        self.conv5 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=5)
        self.conv6 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=6)
        self.conv7 = nn.Conv1d(in_channels=embed_dim, out_channels=256, kernel_size=7)
        
        # Dropout for regularization
        # Randomly zeros out some percentage of the input during the training
        # Prevents overfitting
        self.dropout = nn.Dropout(0.5)
        
        # Final Layer 
        # maps the combined features to the final classification output
        # Input size = num_filters * num_kernels = (256 * 7)
        self.fc = nn.Linear(256*7, num_classes)
        
    def forward(self, x):
        # x shape: [Batch, Max_Len]
        
        emb = self.embedding(x) 
        # emb shape: [Batch, Max_Len, Embed_Dim]
        
        # Conv1d expects [Batch, Channel, Length], so we transpose
        emb = emb.permute(0, 2, 1)
        
        # Apply Convolutions then ReLU activation
        # Result shape: [Batch, Out_Channels, New_Length]
        x1 = torch.relu(self.conv1(emb))
        x2 = torch.relu(self.conv2(emb))
        x3 = torch.relu(self.conv3(emb))
        x4 = torch.relu(self.conv4(emb))
        x5 = torch.relu(self.conv5(emb))
        x6 = torch.relu(self.conv6(emb))
        x7 = torch.relu(self.conv7(emb))
        
        # Max Pooling over time (takes the strongest feature from each map)
        # pooling operation across the length dimension 
        # so dim=2 from [Batch, Out_Channels, New_Length]
        x1 = torch.max(x1, dim=2)[0]
        x2 = torch.max(x2, dim=2)[0]
        x3 = torch.max(x3, dim=2)[0]
        x4 = torch.max(x4, dim=2)[0]
        x5 = torch.max(x5, dim=2)[0]
        x6 = torch.max(x6, dim=2)[0]
        x7 = torch.max(x7, dim=2)[0]
        
        # Concatenate features into a single vector (dim=1)
        # result vector shape: [Batch, Out_Channels * 7] = [Batch, 256*7=1792]
        cat = torch.cat((x1, x2, x3, x4, x5, x6, x7), dim=1)
        
        # Apply dropout to prevent overfitting
        cat = self.dropout(cat)
        
        # Final prediction
        return self.fc(cat)

# Training

In [50]:
import torch.optim as optim
from tqdm import tqdm
from focal_loss import FocalLoss
import os

# Training Function
def train_model(model, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1} (Train)"):
            inputs, labels = inputs.to(device), labels.to(device)
            
            # clear gradients from the previous iteration
            optimizer.zero_grad() 
            # forward
            outputs = model(inputs)
            # calculate loss
            loss = criterion(outputs, labels)
            
            # backpropagation
            loss.backward()
            # adjust the weights based on the gradients
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")
        
        # Validation Step

        # evaluation mode
        # disables dropout it only needed during training
        model.eval() 
        val_loss = 0
        # disable gradient calculation
        # It speeds up the validation process and save memory
        with torch.no_grad(): 
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                # forward
                outputs = model(inputs)
                # calculate loss
                val_loss += criterion(outputs, labels).item()
        print(f"Validation Loss: {val_loss/len(val_loader):.4f}")

# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CNN_model = TextCNN(len(vocab.word2idx), EMBED_DIM, num_classes=6)
CNN_model.to(device)

# Loss and Optimizer

# Binary Cross Entropy Loss is used because we are multi-label classification
# This loss function is apllied to each label independently.
# criterion = nn.BCEWithLogitsLoss() 

# Focal Loss is used to handle class imbalance
# gamma: the focussing parameter that adjusts the rate at which easy examples are down-weighted
# alpha: the class balancing factor
criterion = FocalLoss(gamma=2, alpha=0.75, task_type="multi-label")

# optimization algorithm that adjusts the model's weights 
# during backpropagation to minimize the loss function.
# lr = learning Rate 
optimizer = optim.Adam(CNN_model.parameters(), lr=LEARNING_RATE)
# Run Training or Load Model
# model_save_path = 'textcnn_toxic_classifier.pth'
model_save_path = "textcnn_toxic_classifier_focal_loss.pth"
if os.path.exists(model_save_path):
    print(f"Loading pre-trained model from {model_save_path}")
    state_dict = torch.load(model_save_path, map_location=device)
    CNN_model.load_state_dict(state_dict)
else:
    print("Training new model...")
    train_model(CNN_model, num_epochs=EPOCHS)
    print(f"Saving trained model to {model_save_path}")
    torch.save(CNN_model.state_dict(), model_save_path)
    # save trained model
    MODEL_PATH = 'textcnn_toxic_classifier.pth'
    torch.save(CNN_model.state_dict(), MODEL_PATH)

Loading pre-trained model from textcnn_toxic_classifier_focal_loss.pth


# Prediction using Validation Set

In [51]:
# prediction function
def predict_sentence(text, model=CNN_model, vocab=vocab):
    """Predict the probability of each label for a given text"""
    # evaluation mode
    # disables dropout it only needed during training
    model.eval()
    # disable gradient calculation
    # It speeds up the validation process and save memory
    with torch.no_grad():
        # encode the text to indices
        encoded = vocab.encode(text)
        # convert to tensor
        tensor = torch.tensor([encoded], dtype=torch.long).to(device)
        output = model(tensor)
        # Apply Sigmoid to convert logits to probabilities (0 to 1)
        probs = torch.sigmoid(output).cpu().numpy()[0]

        # probability to label 0.5 threshold
        # if probability is greater than 0.5, it is predicted as 1, otherwise 0
        result = [1 if p > SIGMOID_THRESHOLD else 0 for p in probs]
    
    return dict(zip(['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'], result))

In [52]:
# prediction function
def create_prediction(comments):
    # create prediction datafrme
    pred = pd.DataFrame(columns=val.columns)
    pred['comment_text'] = comments['comment_text'].copy()
    # prediction
    predictions_list_of_dicts = pred['comment_text'].apply(
        lambda text: predict_sentence(text)
    ).tolist()
    # convert list of dictionaries to DataFrame
    predictions_df = pd.DataFrame(predictions_list_of_dicts)
    # update add predictions to pred
    pred[LABEL_COLUMNS] = predictions_df[LABEL_COLUMNS]
    # return pred
    return pred

In [53]:
val_pred = create_prediction(val)

In [54]:
# create confusion matrix and calculate F1 score
from sklearn.metrics import multilabel_confusion_matrix, f1_score, accuracy_score, precision_score, recall_score

def create_confusion_matrix(true, pred):
    """ 
    true: Dataframe that has true labels 
    pred: Dataframe that has predicted labels
    """
    # y_true = true labels
    # y_pred = predicted labels
    y_true = true[LABEL_COLUMNS].values
    y_pred = pred[LABEL_COLUMNS].values

    # The output is a 3D NumPy array: (labels, 2, 2)
    mcm = multilabel_confusion_matrix(y_true, y_pred) 

    # Print the confusion matrix for each label
    print("--- Multilabel Confusion Matrices ---")
    for i, col in enumerate(LABEL_COLUMNS):
        # Extract the 2x2 matrix for the current label
        matrix = mcm[i]
        
        # Components of the confusion matrix (2x2 structure)
        # [[TN, FP], 
        #  [FN, TP]]
        TN = matrix[0, 0]
        FP = matrix[0, 1]
        FN = matrix[1, 0]
        TP = matrix[1, 1]
        
        # Calculate F1 Score for the current label
        # zero_division=0 ensures that if there are no true positives and no predicted positives 
        # (e.g., zero support), F1 is 0.
        f1 = f1_score(y_true[:, i], y_pred[:, i], zero_division=0)

        # other metrics
        acc = accuracy_score(y_true[:, i], y_pred[:, i])
        precision = precision_score(y_true[:, i], y_pred[:, i], zero_division=0)
        recall = recall_score(y_true[:, i], y_pred[:, i], zero_division=0)
        
        # Print the formatted result
        print(f"{col:15} → F1: {f1:.4f}, Acc: {acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f} | TP:{TP} TN:{TN} FP:{FP} FN:{FN}")

create_confusion_matrix(val, val_pred)

--- Multilabel Confusion Matrices ---
toxic           → F1: 0.7472, Acc: 0.9534, Precision: 0.7986, Recall: 0.7020 | TP:1757 TN:22586 FP:443 FN:746
severe_toxic    → F1: 0.3811, Acc: 0.9877, Precision: 0.3789, Recall: 0.3834 | TP:97 TN:25120 FP:159 FN:156
obscene         → F1: 0.7704, Acc: 0.9734, Precision: 0.7185, Recall: 0.8304 | TP:1141 TN:23711 FP:447 FN:233
threat          → F1: 0.3089, Acc: 0.9967, Precision: 0.3585, Recall: 0.2714 | TP:19 TN:25428 FP:34 FN:51
insult          → F1: 0.6721, Acc: 0.9603, Precision: 0.5835, Recall: 0.7924 | TP:1038 TN:23481 FP:741 FN:272
identity_hate   → F1: 0.3427, Acc: 0.9844, Precision: 0.2687, Recall: 0.4727 | TP:104 TN:25029 FP:283 FN:116


# Test the Model with Test Set

In [55]:
test_pred = create_prediction(test)
create_confusion_matrix(test,test_pred)

--- Multilabel Confusion Matrices ---
toxic           → F1: 0.7490, Acc: 0.9549, Precision: 0.8028, Recall: 0.7019 | TP:2145 TN:28332 FP:527 FN:911
severe_toxic    → F1: 0.4144, Acc: 0.9888, Precision: 0.4349, Recall: 0.3956 | TP:127 TN:31429 FP:165 FN:194
obscene         → F1: 0.7660, Acc: 0.9730, Precision: 0.7166, Recall: 0.8227 | TP:1411 TN:29642 FP:558 FN:304
threat          → F1: 0.3200, Acc: 0.9968, Precision: 0.3158, Recall: 0.3243 | TP:24 TN:31789 FP:52 FN:50
insult          → F1: 0.6721, Acc: 0.9609, Precision: 0.5838, Recall: 0.7918 | TP:1278 TN:29390 FP:911 FN:336
identity_hate   → F1: 0.3401, Acc: 0.9836, Precision: 0.2700, Recall: 0.4592 | TP:135 TN:31256 FP:365 FN:159
